# Assignment 05 · Notebook 05: Diabetes, bốn mô hình Conv1D

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Cùng bốn cơ chế của notebook 03 (gốc, VGG, ResNet, SE-ResNet) nhưng đổi `Conv2d` thành `Conv1d`:
mỗi bệnh nhân là một vectơ đặc trưng $x \in \mathbb{R}^{d}$, đưa vào mạng dưới dạng tensor
`[B, 1, d]` để bộ lọc trượt dọc các đặc trưng. Đầu ra là một logit, hàm mất mát
`BCEWithLogitsLoss` có `pos_weight` = số âm / số dương vì lớp dương chỉ khoảng 8,8%.

Mô hình dưới 70 000 tham số nên chạy trên **CPU** (8 luồng): với lô nhỏ như vậy, chi phí
chuyển dữ liệu lên GPU lớn hơn phần tính toán.

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import torch
from a05.data import prepare_diabetes
from a05.experiment import run_tabular_experiment
from a05.models import MODEL_LABELS
from a05.train import TAB_CFG

print("Thiết bị: CPU ·", TAB_CFG["threads"], "luồng · PyTorch", torch.__version__)
print("Cấu hình:", TAB_CFG)

Thiết bị: CPU · 8 luồng · PyTorch 2.13.0+cu126
Cấu hình: {'epochs': 30, 'batch_size': 512, 'lr': 0.001, 'weight_decay': 0.0001, 'patience': 5, 'threads': 8}


In [2]:
D = prepare_diabetes()
print("Dòng sau khi bỏ trùng:", D["n_rows"], "· số dòng trùng đã bỏ:", D["n_duplicates"])
print("Số đặc trưng sau mã hoá:", len(D["feature_names"]))
print(D["feature_names"])
for s in ("train", "val", "test"):
    print(f"{s:5s}", D["X"][s].shape, "tỉ lệ dương", round(float(D["y"][s].mean()), 4))

Dòng sau khi bỏ trùng: 96146 · số dòng trùng đã bỏ: 3854
Số đặc trưng sau mã hoá: 15
['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'hypertension', 'heart_disease', 'gender=Female', 'gender=Male', 'gender=Other', 'smoking_history=No Info', 'smoking_history=current', 'smoking_history=ever', 'smoking_history=former', 'smoking_history=never', 'smoking_history=not current']
train (67302, 15) tỉ lệ dương 0.0882
val   (14422, 15) tỉ lệ dương 0.0883
test  (14422, 15) tỉ lệ dương 0.0882


## Huấn luyện

AdamW (lr 1e-3, weight decay 1e-4), lô 512, tối đa 30 epoch, dừng sớm khi val loss không giảm
sau 5 epoch. Ngưỡng phân loại được chọn trên **validation** sao cho F1 lớn nhất rồi mới áp cho test.

In [3]:
results = run_tabular_experiment(D)

== diabetes / basic


  epoch  1  train 0.5202/0.8412  val 0.4542/0.8565  0.8s


  epoch  2  train 0.4016/0.8842  val 0.4298/0.8814  0.7s


  epoch  3  train 0.3824/0.8899  val 0.4209/0.8586  0.7s


  epoch  4  train 0.3729/0.8885  val 0.4100/0.8701  0.7s


...

  epoch 10  train 0.3200/0.8987  val 0.4019/0.9107  2.8s


  epoch 11  train 0.3197/0.9007  val 0.3861/0.8852  2.8s


  epoch 12  train 0.3167/0.8999  val 0.3972/0.8926  2.8s


  epoch 13  train 0.3111/0.9003  val 0.3874/0.8796  2.7s
  dừng sớm ở epoch 13, tốt nhất là epoch 8


   test F1 0.8128  ROC-AUC 0.9765  ngưỡng 0.872


In [4]:
df = pd.DataFrame(results).T.drop(columns=["confusion"])
df.index = [MODEL_LABELS[n] for n in df.index]
df[["acc", "precision", "recall", "f1", "roc_auc", "pr_auc", "threshold", "params", "epoch_s", "best_epoch"]].round(4)

,acc,precision,recall,f1,roc_auc,pr_auc,threshold,params,epoch_s,best_epoch
BasicCNN,0.96748,0.883477,0.727201,0.797758,0.976811,0.880628,0.87935,12065,0.595548,22
VGGNet,0.968728,0.910911,0.715409,0.801409,0.975191,0.877889,0.897652,57073,1.939873,6
ResNet,0.968867,0.916077,0.712264,0.801415,0.976085,0.879846,0.869249,59873,2.148604,4
SE-ResNet,0.971225,0.953439,0.708333,0.81281,0.976471,0.881562,0.871923,61913,2.836792,8
